# Quick Start: Synthetic Control with CausalPy

This notebook runs a complete synthetic control analysis in under 2 minutes.
Copy-paste this as a starting template for your own SC study.

**What you need:**
- Panel data: one treated unit + multiple untreated donor units
- A known intervention time (when the treatment started)
- At least 10 pre-intervention observations per unit

**What you get:**
- Posterior donor weights
- Counterfactual trajectory with 94% HDI
- Permutation p-value from in-space placebo tests

---

In [ ]:
import sys; sys.path.insert(0, '../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import causalpy as cp
import arviz as az
from scipy.optimize import minimize

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

## Step 1: Prepare Your Panel Data

Data must be in **long format**: one row per (unit, time) combination.
The treated unit's post-intervention outcome contains the observed (potentially treated) values.

In [ ]:
# --- REPLACE THIS SECTION WITH YOUR DATA ---
# Synthetic panel: one treated state + 15 donor states, 25 years

N_YEARS = 25
N_PRE = 15        # Intervention at year 15
N_POST = 10
N_DONORS = 15
TRUE_EFFECT = -18.0  # Immediate level change

years = np.arange(N_YEARS)

# Donor trajectories
np.random.seed(42)
donor_baselines = np.random.uniform(90, 130, N_DONORS)
Y_donors = np.zeros((N_YEARS, N_DONORS))
for j in range(N_DONORS):
    Y_donors[:, j] = (
        donor_baselines[j] - 1.0 * years + np.random.normal(0, 3, N_YEARS)
    )

# Treated unit: weighted combination of 3 donors + treatment effect
true_weights = np.zeros(N_DONORS)
true_weights[0] = 0.50
true_weights[4] = 0.30
true_weights[9] = 0.20

y_treated_latent = Y_donors @ true_weights + np.random.normal(0, 1.5, N_YEARS)
y_treated = y_treated_latent.copy()
for t in range(N_PRE, N_YEARS):
    y_treated[t] += TRUE_EFFECT

# Build long-format DataFrame
donor_names = [f"Donor_{j:02d}" for j in range(N_DONORS)]
rows = []
for t in range(N_YEARS):
    rows.append({"unit": "TreatedUnit", "year": t, "outcome": y_treated[t]})
for j, name in enumerate(donor_names):
    for t in range(N_YEARS):
        rows.append({"unit": name, "year": t, "outcome": Y_donors[t, j]})

panel_df = pd.DataFrame(rows)
print(f"Panel shape: {panel_df.shape}")
print(f"Units: {panel_df['unit'].nunique()} ({N_DONORS} donors + 1 treated)")
print(f"Years: {panel_df['year'].nunique()} ({N_PRE} pre + {N_POST} post)")
print(panel_df.head(3))

## Step 2: Fit the Bayesian Synthetic Control

In [ ]:
# Build the formula: outcome regressed on each donor unit
donor_formula_terms = " + ".join(donor_names)
formula = f"outcome ~ 0 + {donor_formula_terms}"

sc_model = cp.pymc_experiments.SyntheticControl(
    data=panel_df,
    treatment_time=N_PRE,
    formula=formula,
    group_variable_name="unit",
    treated_group="TreatedUnit",
    model=cp.pymc_models.WeightedSumFitter(
        sample_kwargs={
            "draws": 1000,
            "tune": 1000,
            "chains": 4,
            "random_seed": 42,
        }
    ),
)

print("Synthetic control model fitted.")

## Step 3: Check Pre-Period Fit

In [ ]:
# Extract posterior weight means and compute synthetic trajectory
beta_posterior = sc_model.idata.posterior["beta"].values.reshape(-1, N_DONORS)
weight_means = beta_posterior.mean(axis=0)

# Synthetic trajectory using posterior mean weights
y_synthetic = Y_donors @ weight_means  # shape (N_YEARS,)

# Pre-period fit quality
pre_residuals = y_treated[:N_PRE] - y_synthetic[:N_PRE]
rmspe_pre = np.sqrt(np.mean(pre_residuals ** 2))
pre_std = y_treated[:N_PRE].std()
rmspe_pct = rmspe_pre / pre_std

print("Pre-Period Fit Assessment")
print(f"  RMSPE:            {rmspe_pre:.2f}")
print(f"  Pre-period std:   {pre_std:.2f}")
print(f"  RMSPE / std:      {rmspe_pct:.1%}")
print()
if rmspe_pct < 0.20:
    print("  GOOD FIT: RMSPE < 20% of pre-period std.")
elif rmspe_pct < 0.35:
    print("  MODERATE FIT: Consider expanding donor pool or adding predictors.")
else:
    print("  POOR FIT: Synthetic control may not be credible. Revise donor pool.")

# Top donors by weight
top_idx = np.argsort(weight_means)[::-1][:5]
print()
print("Top 5 donors by weight:")
for i in top_idx:
    print(f"  {donor_names[i]}: {weight_means[i]:.3f}  (true: {true_weights[i]:.2f})")

## Step 4: Visualize the Counterfactual

In [ ]:
# Posterior predictive counterfactual with uncertainty
cf_samples = beta_posterior @ Y_donors.T  # (n_samples, N_YEARS)
cf_mean = cf_samples.mean(axis=0)
cf_lower = np.percentile(cf_samples, 3, axis=0)
cf_upper = np.percentile(cf_samples, 97, axis=0)

gap_samples = y_treated[np.newaxis, :] - cf_samples
gap_mean = gap_samples.mean(axis=0)
gap_lower = np.percentile(gap_samples, 3, axis=0)
gap_upper = np.percentile(gap_samples, 97, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: trajectory
ax = axes[0]
ax.fill_between(years, cf_lower, cf_upper, alpha=0.2, color="#e74c3c")
ax.plot(years, cf_mean, color="#e74c3c", linewidth=2, linestyle="--",
        label="Synthetic (mean ± 94% HDI)")
ax.plot(years, y_treated, color="#2980b9", linewidth=2.5, label="Treated unit (observed)")
ax.axvline(N_PRE, color="#27ae60", linestyle=":", linewidth=2, label="Intervention")
ax.set_title("Treated vs Synthetic Counterfactual", fontsize=12)
ax.set_xlabel("Year")
ax.set_ylabel("Outcome")
ax.legend(fontsize=10)

# Right: gap
ax = axes[1]
ax.fill_between(years[N_PRE:], gap_lower[N_PRE:], gap_upper[N_PRE:],
                alpha=0.3, color="#e74c3c", label="94% HDI")
ax.plot(years[:N_PRE], gap_mean[:N_PRE], color="#aaaaaa", linewidth=1.5)
ax.plot(years[N_PRE:], gap_mean[N_PRE:], color="#e74c3c", linewidth=2.5,
        label="Causal effect (mean)")
ax.axhline(TRUE_EFFECT, color="#27ae60", linestyle=":", linewidth=2,
           label=f"True effect ({TRUE_EFFECT}")
ax.axhline(0, color="black", linestyle="--", linewidth=1.5)
ax.axvline(N_PRE, color="#27ae60", linestyle=":", linewidth=2)
ax.set_title("Causal Effect Gap with 94% HDI", fontsize=12)
ax.set_xlabel("Year")
ax.set_ylabel("Gap (Treated − Synthetic)")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

## Step 5: Permutation P-Value (Quick Version)

A simplified in-space placebo test using classical SC weights.

In [ ]:
# Quick permutation test: classical SC for all units

Y_all = np.column_stack([y_treated, Y_donors])  # (N_YEARS, N_DONORS+1)
n_units = Y_all.shape[1]
TREATED_IDX = 0

def quick_sc_weights(y_pre, Y_d_pre):
    n_d = Y_d_pre.shape[1]
    res = minimize(
        lambda w: np.sum((y_pre - Y_d_pre @ w) ** 2),
        np.ones(n_d) / n_d,
        method="SLSQP",
        bounds=[(0, 1)] * n_d,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
    )
    return res.x


rmspe_pre_all = np.zeros(n_units)
rmspe_post_all = np.zeros(n_units)

print("Running placebo tests...")
for j in range(n_units):
    y_j_pre = Y_all[:N_PRE, j]
    y_j_post = Y_all[N_PRE:, j]
    donors_j = [k for k in range(n_units) if k != j]
    Y_d_pre = Y_all[:N_PRE, :][:, donors_j]
    Y_d_post = Y_all[N_PRE:, :][:, donors_j]

    w = quick_sc_weights(y_j_pre, Y_d_pre)
    rmspe_pre_all[j] = np.sqrt(np.mean((y_j_pre - Y_d_pre @ w) ** 2))
    rmspe_post_all[j] = np.sqrt(np.mean((y_j_post - Y_d_post @ w) ** 2))

ratio = rmspe_post_all / rmspe_pre_all
p_value = (ratio >= ratio[TREATED_IDX]).mean()
rank = int((ratio >= ratio[TREATED_IDX]).sum())

print(f"\nPermutation p-value: {p_value:.4f}")
print(f"Treated unit rank:   {rank} of {n_units} (1 = largest gap)")
print(f"Min achievable p:    {1/n_units:.4f}")
print()
if p_value < 0.05:
    print("Significant at 5%: effect is distinguishable from placebo distribution.")
elif p_value < 0.10:
    print("Marginally significant at 10%.")
else:
    print(f"Not significant. Donor pool may be too small ({n_units} units, min p = {1/n_units:.2f}).")

## Quick Reference

### Data requirements
- Long format: columns `[unit, time, outcome]`
- Treated unit name must match `treated_group` argument
- All units must have the same time points

### Validity checks
| Check | Tool | Pass criterion |
|-------|------|----------------|
| Pre-period fit | RMSPE / pre-period std | < 20% |
| Donor comparability | Weight sparsity | Few donors with high weight |
| Statistical significance | Permutation p-value | < 0.05 (if donor pool large enough) |
| Pre-period validity | In-time placebo | Small gap at placebo date |

### Next steps
- For full placebo tests with spaghetti plot: Module 03, Notebook 02
- For Bayesian weight uncertainty: Module 03, Notebook 03
- For ITS (single unit): see `quickstart_its.ipynb`

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])